# Design matrix — one-camera setup

Single (left) camera variant of `../2_design_matrix.ipynb`.

Differences from the two-camera version:
- Only the left camera is loaded; both paws are read from `leftCamera`.
- Licks come from `get_left_licks` (left camera only) instead of `merge_licks`.
- No `avg_whisker_me` column (needs a second camera).
- One-camera videos can be recorded at **30 or 60 Hz**: the native rate is
  detected per session with `get_frame_rate`, the output grid is fixed at
  **fs = 30 Hz** for cross-session consistency, and an anti-aliasing low-pass
  is applied **only** when a session is downsampled (native > 30 Hz).

In [1]:
import os
import numpy as np
import pickle
import pandas as pd
from brainbox.io.one import SessionLoader
import gc
import concurrent.futures
import gzip

from segmentation_functions import get_left_licks, resample_common_time, interpolate_nans, low_pass, get_frame_rate
from one.api import ONE
one = ONE(mode='remote')

In [2]:
prefix = '/home/ines/repositories/'

In [3]:
data_path = '/home/ines/repositories/representation_learning_variability/paper-individuality/segmentation/1_camera_setup/temp_data/design_matrices/' # data should be moved to the drive manually
data_path = '/home/ines/repositories/representation_learning_variability/paper-individuality/data/design_matrices/1_camera_setup/session_1_individuality/' # data should be moved to the drive manually
sessions = list(pd.read_csv('nm_filtered_eids.csv')['eid'])
sessions = list(pd.read_csv('lda_first_training_eids.csv')['eid'])

In [13]:

sessions_to_process = []
for m, mat in enumerate(sessions):
    file_path = one.eid2path(mat)
    
    if prefix == '/home/ines/repositories/':
        mouse_name = file_path.parts[8]
    else:
        mouse_name = file_path.parts[7]
    result_filename = data_path + "design_matrix_" + str(mat) + '_'  + mouse_name
    if not os.path.exists(result_filename):
        sessions_to_process.append(mat)

print(f"Found {len(sessions_to_process)} sessions to process.")

Found 47 sessions to process.


In [14]:
def process_design_matrix(session):

    file_path = one.eid2path(session)
    if prefix == '/home/ines/repositories/':
        mouse_name = file_path.parts[8]
    else:
        mouse_name = file_path.parts[7]
    try:
        
        """ LOAD VARIABLES """
        sl = SessionLoader(eid=session, one=one)
        sl.load_pose(views=['left'], tracker='lightningPose')
        sl.load_session_data(trials=True, wheel=True, motion_energy=True)

        # Check if all data is available
        # if np.sum(sl.data_info['is_loaded']) >= 4:

        # Poses
        poses = sl.pose
        lc_t = np.asarray(poses['leftCamera']['times'])
        # One-camera videos can be recorded at 30 or 60 Hz; detect the native rate
        left_fr = get_frame_rate(lc_t)

        # Motion energy (anti-alias only when downsampling a 60 Hz session to the 30 Hz grid)
        me = sl.motion_energy
        mel_t = lc_t
        if left_fr > 30:
            motion_energy_l = low_pass(interpolate_nans(me['leftCamera']['whiskerMotionEnergy'], left_fr),
                                    cutoff=12, sf=left_fr)
        else:
            motion_energy_l = interpolate_nans(me['leftCamera']['whiskerMotionEnergy'], left_fr)

        # Licks
        features = ['tongue_end_l_x', 'tongue_end_l_y', 'tongue_end_r_x', 'tongue_end_r_y']
        lick_t, licks = get_left_licks(poses, features, common_fs=60)

        # Paws (a single camera sees both paws)
        if left_fr > 30:
            l_paw_x = low_pass(interpolate_nans(poses['leftCamera']['paw_r_x'], left_fr), cutoff=12, sf=left_fr)
            l_paw_y = low_pass(interpolate_nans(poses['leftCamera']['paw_r_y'], left_fr), cutoff=12, sf=left_fr)
            r_paw_x = low_pass(interpolate_nans(poses['leftCamera']['paw_l_x'], left_fr), cutoff=12, sf=left_fr)
            r_paw_y = low_pass(interpolate_nans(poses['leftCamera']['paw_l_y'], left_fr), cutoff=12, sf=left_fr)
        else:
            l_paw_x = interpolate_nans(poses['leftCamera']['paw_r_x'], left_fr)
            l_paw_y = interpolate_nans(poses['leftCamera']['paw_r_y'], left_fr)
            r_paw_x = interpolate_nans(poses['leftCamera']['paw_l_x'], left_fr)
            r_paw_y = interpolate_nans(poses['leftCamera']['paw_l_y'], left_fr)
        l_paw_t = lc_t
        r_paw_t = lc_t

        # Wheel
        wheel = sl.wheel
        wheel_t = np.asarray(wheel['times'], dtype=np.float64)
        wheel_vel = wheel['velocity'].astype(np.float32)

        # Common resampling window
        onset = max(lc_t.min(), wheel_t.min(), lick_t.min())
        offset = min(lc_t.max(), wheel_t.max(), lick_t.max())
        fs = 30
        ref_t = np.arange(onset, offset, 1 / fs, dtype=np.float64)

        # Restrict to time window
        def restrict(t, x):
            mask = (t >= onset) & (t <= offset)
            return t[mask], x[mask]

        mel_t, motion_energy_l = restrict(mel_t, motion_energy_l)
        wheel_t, wheel_vel = restrict(wheel_t, wheel_vel)
        l_paw_t_x, l_paw_x = restrict(l_paw_t, l_paw_x)
        l_paw_t_y, l_paw_y = restrict(l_paw_t, l_paw_y)
        r_paw_t_x, r_paw_x = restrict(r_paw_t, r_paw_x)
        r_paw_t_y, r_paw_y = restrict(r_paw_t, r_paw_y)
        lick_t, licks = restrict(lick_t, licks)

        # Resample
        mel_down, rt = resample_common_time(ref_t, mel_t, motion_energy_l, kind='linear')
        wh_down, _ = resample_common_time(ref_t, wheel_t, wheel_vel, kind='linear')
        lk_down, _ = resample_common_time(ref_t, lick_t, licks, kind='nearest')
        lpx_down, _ = resample_common_time(ref_t, l_paw_t_x, l_paw_x, kind='linear')
        lpy_down, _ = resample_common_time(ref_t, l_paw_t_y, l_paw_y, kind='linear')
        rpx_down, _ = resample_common_time(ref_t, r_paw_t_x, r_paw_x, kind='linear')
        rpy_down, _ = resample_common_time(ref_t, r_paw_t_y, r_paw_y, kind='linear')

        # Create design matrix
        design_matrix = pd.DataFrame({
            'Bin': rt,
            'Lick count': lk_down.astype(np.int8),
            'avg_wheel_vel': wh_down,
            'whisker_me': mel_down,
            'l_paw_x': lpx_down,
            'l_paw_y': lpy_down,
            'r_paw_x': rpx_down,
            'r_paw_y': rpy_down,
        })

        # """ LOAD TRIAL DATA """
        session_trials = sl.trials
        session_start = list(session_trials['goCueTrigger_times'])[0]
        design_matrix = design_matrix.loc[(design_matrix['Bin'] > session_start)]

        """ SAVE DATA """
        # Save unnormalized design matrix
        filename = data_path + "design_matrix_" + str(session) + '_'  + mouse_name
        design_matrix.to_parquet(filename, compression='gzip')

        # Save trials
        filename = data_path + "session_trials_" + str(session) + '_'  + mouse_name
        session_trials.to_parquet(filename, compression='gzip')

        del design_matrix, session_trials, sl
        gc.collect()

        # else:
        #     print('Data missing for session '+session)
            
    except Exception as e:
        print(e, session)


def parallel_process_data(sessions, function_name):
    with concurrent.futures.ThreadPoolExecutor() as executor:

        # Process each chunk in parallel
        executor.map(function_name, sessions)

In [15]:
for s, session in enumerate(sessions_to_process[3:]):
    process_design_matrix(session)

2026-07-15 13:47:33 INFO     one.py:1288 Loading trials data


/home/ines/miniconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:436: ALFWarning: Multiple revisions: "", "2024-07-15"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


2026-07-15 13:47:33 INFO     one.py:1288 Loading wheel data


(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/danlab/Subjects/DY_011/2019-11-21/001/alf/_ibl_wheel.position.npy: 100%|██████████| 723k/723k [00:16<00:00, 45.1kB/s]
(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/danlab/Subjects/DY_011/2019-11-21/001/alf/_ibl_wheel.timestamps.npy: 100%|██████████| 723k/723k [00:13<00:00, 55.1kB/s]

2026-07-15 13:48:03 INFO     one.py:1288 Loading motion_energy data


...

2026-07-15 14:15:48 INFO     one.py:1288 Loading wheel data
2026-07-15 14:15:48 INFO     one.py:1288 Loading motion_energy data


(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/steinmetzlab/Subjects/NR_0027/2022-06-02/001/alf/leftCamera.ROIMotionEnergy.npy: 100%|██████████| 741k/741k [00:22<00:00, 33.2kB/s]


2026-07-15 14:16:13 WARNING  one.py:1292 Could not load motion_energy data.
2026-07-15 14:16:13 INFO     one.py:1288 Loading pupil data
2026-07-15 14:16:13 INFO     one.py:1450 Pupil diameter not available, trying to compute on the fly.
2026-07-15 14:16:13 WARNING  one.py:1292 Could not load pupil data.
'leftCamera' 167f98be-6704-430f-b531-2d4ab2aae932


(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/angelakilab/Subjects/NYU-30/2020-07-17/001/alf/_ibl_leftCamera.times.npy: 100%|██████████| 2.25M/2.25M [01:08<00:00, 32.8kB/s]
(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/angelakilab/Subjects/NYU-30/2020-07-17/001/alf/_ibl_leftCamera.lightningPose.pqt:  15%|█▍        | 19.9M/137M [01:06<06:30, 300kB/s]  


KeyboardInterrupt: 